# fedOPD data fetch

Three datasets, each in its own folder under `../data`:

| # | dataset | folder |
|---|---------|--------|
| 1 | NHS RTT monthly Full CSV zips, Apr 2018 – Mar 2026 (96 files, ~8 GB extracted) | `data/rtt_zips` -> `data/rtt_csv` |
| 2 | Trust -> ICB / geography reference (ODS ORD API) | `data/reference` |
| 3 | KH03 overnight bed availability & occupancy | `data/kh03` |



In [1]:
import re                                  # regex — pulls the month token out of filenames
import zipfile                             # unzip the downloaded monthly archives
from pathlib import Path                   # filesystem paths
from datetime import datetime              # parse 'Mar19' so months sort chronologically

import requests                            # all HTTP: page scrapes, file downloads, ODS API
import pandas as pd                        # builds the trust -> ICB table at the end
from bs4 import BeautifulSoup              # parses the NHS html pages to find download links

# folder layout: one folder per dataset, everything under ../data
DATA = Path("../data")                     # repo-root/data, relative to this notebook in src/
RTT_ZIPS = DATA / "rtt_zips"               # dataset 1a: raw monthly zips as downloaded (~330 MB)
RTT_CSV = DATA / "rtt_csv"                 # dataset 1b: csvs extracted from those zips (~7.7 GB)
REF = DATA / "reference"                   # dataset 2: trust -> ICB lookup table
KH03 = DATA / "kh03"                       # dataset 3: quarterly bed spreadsheets
for d in (RTT_ZIPS, RTT_CSV, REF, KH03):   # create all four up front
    d.mkdir(parents=True, exist_ok=True)   # parents=True builds ../data itself; exist_ok = no-op on rerun

# nhs.uk returns 403 to the default python-requests user agent, so pretend to be a browser
HEADERS = {"User-Agent": "Mozilla/5.0"}    # reused by every request in this notebook

## 1. Build the RTT Full CSV manifest

Each financial year (2018-19 ... 2025-26) has one page listing that year's monthly files. We scrape all eight pages, keep only links whose filename says Full-CSV and ends in .zip, and key them by the month token in the filename (e.g. `Mar19`). When a month appears twice (original + revised), the revised link wins.

Cross-checked: this filter picks exactly the official links, e.g. `Full-CSV-data-file-Mar19-revised-ZIP-3506K.zip` and `Full-CSV-data-file-Mar26-ZIP-3M-revised.zip`. Ignore the size hint in the filename — the Mar22 revised file says '111805K' but is actually ~4.5 MB on the server; sizes vary wildly month to month and that's fine.

In [2]:
# build '2018-19', '2019-20', ..., '2025-26' — these match the url slug of each year page
YEARS = [f"{y}-{str(y + 1)[2:]}" for y in range(2018, 2026)]   # str(2019)[2:] -> '19'

manifest = {}                                          # month key like 'Mar19' -> download url
for yr in YEARS:                                       # one page per financial year, 8 total
    page = f"https://www.england.nhs.uk/statistics/statistical-work-areas/rtt-waiting-times/rtt-data-{yr}/"
    soup = BeautifulSoup(requests.get(page, headers=HEADERS).text, "html.parser")   # fetch + parse html
    for a in soup.find_all("a", href=True):            # walk every <a href=...> link on the page
        href = a["href"]                               # full url
        name = href.split("/")[-1]                     # just the filename part of the url
        if "full-csv" not in name.lower() or not name.endswith(".zip"):
            continue                                   # skip XLS views, commissioner files, everything else
        # filenames look like 'Full-CSV-data-file-Mar19-revised-ZIP-3506K.zip'
        # the regex grabs 'Mar19': 3 letters (upper then 2 lower) followed by 2 digits
        key = re.search(r"([A-Z][a-z]{2}\d{2})", name).group(1)
        # first sighting of a month goes in; a 'revised' filename then overwrites the original
        if key not in manifest or "revised" in name.lower():
            manifest[key] = href                       # last revised link for that month wins

# sort chronologically ('Apr18' ... 'Mar26'); %b%y parses 'Mar19' into a real date to sort on
months = sorted(manifest, key=lambda k: datetime.strptime(k, "%b%y"))
print(len(months), "months:", months[0], "->", months[-1])   # expect: 96 months: Apr18 -> Mar26

96 months: Apr18 -> Mar26


## 2. Download all 96 zips

Streamed in 1 MB chunks so big files don't sit in memory. Each file downloads to a `.part` name and is renamed only once fully written — so a completed filename on disk always means a complete download, and rerunning the cell skips finished files and resumes the rest. File sizes vary a lot month to month; that's expected.

In [3]:
for k in months:                                   # chronological order, Apr18 -> Mar26
    url = manifest[k]                              # the (revised, if any) url for this month
    dest = RTT_ZIPS / url.split("/")[-1]           # save under the server's own filename
    if dest.exists():
        continue                                   # already fully downloaded on a previous run
    tmp = dest.with_suffix(".part")                # write in progress under a temp name
    r = requests.get(url, headers=HEADERS, stream=True)   # stream=True: don't load it all into RAM
    with open(tmp, "wb") as f:                     # binary write
        for chunk in r.iter_content(1 << 20):      # 1 << 20 = 1 MB at a time
            f.write(chunk)                         # append each chunk to the .part file
    tmp.rename(dest)                               # rename only after the last byte -> final name = complete
    print(k, dest.name, f"{dest.stat().st_size / 1e6:.1f} MB")   # per-month progress line

print("done,", len(list(RTT_ZIPS.glob('*.zip'))), "zips on disk")   # expect 96

done, 96 zips on disk


## 3. Extract every zip into one csv folder

In [4]:
# each zip holds exactly that month's full extract csv; dump them all into one flat folder
for z in sorted(RTT_ZIPS.glob("*.zip")):           # every zip we just downloaded
    zipfile.ZipFile(z).extractall(RTT_CSV)         # extract members into data/rtt_csv

print(len(list(RTT_CSV.glob('*.csv'))), "csvs extracted")   # expect 96, ~7.7 GB total

96 csvs extracted


## 4. Trust -> ICB reference (ODS ORD API)

Two steps: one call lists every organisation with primary role RO197 (= NHS Trust), then one call per trust fetches its full record, from which we take the active RE5 ('is commissioned by') relationship — its target is the trust's ICB, which carries primary role **RO261** in ODS. Verified live: e.g. North London NHS FT -> `Z9B2Z` (NHS West and North London ICB).

The list returns ~550 trusts of which ~250 are Active; we keep the inactive ones too since they appear in the 2018-era RTT months (their `icb_code` stays empty). ~550 small requests, a couple of minutes total.

In [5]:
ORD = "https://directory.spineservices.nhs.uk/ORD/2-0-0"   # NHS ODS organisation reference API

# every org whose PRIMARY role is RO197 (= NHS Trust), current and historic, in one page
trusts = requests.get(
    f"{ORD}/organisations",                                # the list endpoint
    params={"PrimaryRoleId": "RO197",                      # RO197 = NHS TRUST
            "Limit": 1000,                                 # ~550 exist, so one page covers it
            "_format": "json"},                            # else it returns xml
    headers=HEADERS,
).json()["Organisations"]                                  # unwrap the json envelope
print(len(trusts), "trusts")                               # expect ~544

544 trusts


In [6]:
rows = []                                          # one dict per trust, becomes the dataframe
for t in trusts:                                   # ~544 trusts -> ~544 requests, a couple of minutes
    code = t["OrgId"]                              # provider code, e.g. 'R0A' — same code as in the RTT csvs
    # the list endpoint gives name/status only; fetch the full record for rels + geography
    rec = requests.get(f"{ORD}/organisations/{code}", headers=HEADERS).json()["Organisation"]
    icb_code = ""                                  # default: no ICB (true for long-dissolved trusts)
    for rel in rec.get("Rels", {}).get("Rel", []):  # .get twice: some orgs have no Rels block at all
        tgt = rel["Target"]                        # the org on the other end of the relationship
        # RE5 = 'is commissioned by'; keep only the live one whose target is an ICB (role RO261)
        if rel["Status"] == "Active" and rel["id"] == "RE5" and tgt["PrimaryRoleId"]["id"] == "RO261":
            icb_code = tgt["OrgId"]["extension"]   # e.g. 'QOP' — the ICB's own ods code
    rows.append({
        "trust_code": code,                        # join key to the RTT 'Provider Org Code' column
        "trust_name": t["Name"],
        "status": t["Status"],                     # Active / Inactive (old trusts merged away)
        "icb_code": icb_code,                      # empty for dissolved trusts — expected, not an error
        "postcode": rec["GeoLoc"]["Location"].get("PostCode", ""),   # for the geographic region tier
    })

ref = pd.DataFrame(rows)                           # ~544 rows x 5 cols
ref.to_csv(REF / "trust_icb.csv", index=False)     # -> data/reference/trust_icb.csv
ref.head()                                         # peek at the first 5

,trust_code,trust_name,status,icb_code,postcode
0,G6V2S,NORTH LONDON NHS FOUNDATION TRUST,Active,Z9B2Z,NW1 0PE
1,K0N6A,THE ONLINE NHS TRUST,Active,QKK,SE1 8UG
2,R0A,MANCHESTER UNIVERSITY NHS FOUNDATION TRUST,Active,QOP,M13 9WL
3,R0B,SOUTH TYNESIDE AND SUNDERLAND NHS FOUNDATION T...,Active,QHM,SR4 7TP
4,R0C,PROJECT NIGHTINGALE NHS TRUST,Inactive,,E16 1XL


## 5. KH03 overnight beds 

The bed-availability page links every quarterly overnight spreadsheet back to 2010 — 61 files as of Jul 2026, latest `Beds-Open-Overnight-Web_File-Q4-2025-26.xlsx` (verified live). We take any xls/xlsx link with 'night' in the filename and download whatever isn't already on disk.

In [7]:
page = "https://www.england.nhs.uk/statistics/statistical-work-areas/bed-availability-and-occupancy/bed-data-overnight/"
soup = BeautifulSoup(requests.get(page, headers=HEADERS).text, "html.parser")   # fetch + parse the page

# {...} is a set comprehension, so repeated links on the page collapse to one entry
links = sorted({
    a["href"] for a in soup.find_all("a", href=True)                    # every link on the page
    if a["href"].split("/")[-1].lower().endswith((".xls", ".xlsx"))     # spreadsheets only
    and "night" in a["href"].split("/")[-1].lower()                     # 'overnight', not day-case beds
})
print(len(links), "files")                          # expect 61, back to 2011

for url in links:
    dest = KH03 / url.split("/")[-1]                # keep the server's filename
    if dest.exists():
        continue                                    # already downloaded on a previous run
    dest.write_bytes(requests.get(url, headers=HEADERS).content)   # few hundred KB each, no streaming needed
    print(dest.name)

61 files
